In [1]:
%matplotlib widget
%load_ext autoreload
%autoreload 2

# nQ Intensity Optimizer — Demo

*Created by Claude (Anthropic), 2026-03-03.*

This notebook demonstrates `optimize_nQ_intensities` from `nQ_optimizer.py`.

The function finds the laser pulse intensities that minimize the error when
extracting orders of nQ or TA signals from intensity-dependent
nonlinear spectroscopy measurements.

**Key experimental inputs:**
- `noise_over_S_max` = σ / S_max  (dimensionless noise level)
- `I_sat`   — saturation intensity (any physical unit, e.g. nJ)
- `I_max`   — maximum achievable intensity (same unit as `I_sat`)

**Your choices:**
- `ns`  — which nQ signals to evaluate, e.g. `[1]` or `[1, 2]`
- `Ns`  — how many distinct pulse intensities to compare, e.g. `[3, 4, 5]`
- `M`   — number of perturbation orders to include in the error
- `get_model` — saturation model factory (built-in or user-defined)

In [ ]:
import os, sys
# Ensure order_separation.py and nQ_optimizer.py are importable
notebook_dir = os.path.dirname(os.path.abspath('nQ_optimizer_demo.ipynb'))
sys.path.insert(0, notebook_dir)

import numpy as np
from nQ_optimizer import (
    nQ_exponential_saturation_model,
    nQ_saturable_absorption_model,
    ta_exponential_model,
    ta_saturable_absorption_model,
    optimize_nQ_intensities,
)

## Example 1: Exponential saturation model, optimizing over N

Use `nQ_exponential_saturation_model` (exponential TA saturation → nQ Bessel-function signal).
Optimize 1Q and 2Q for N = 3, 4, 5, 6 intensities.

The function returns three arrays and optionally prints tables:
- `total_err`: combined/random/systematic error for each (N, nQ) pair
- `optimal_I`: optimal intensities in physical units
- `err_by_order`: per-order breakdown (only when `len(Ns) == 1`)

In [ ]:
# --- Experimental parameters ---
I_sat          = 45.7 / 60   # saturation intensity (dimensionless here; could be nJ)
I_max          = 1.0          # maximum achievable intensity
S_max          = 0.0543       # TA signal at saturation
noise_measured = 5e-7         # measured noise amplitude

# --- User choices ---
ns = [1, 2]        # evaluate 1Q and 2Q signals
Ns = [3, 4, 5, 6]  # compare 3-, 4-, 5-, and 6-intensity schemes
M  = 3             # include orders 3, 5, 7 (i.e. M perturbation orders)

total_err, optimal_I, err_by_order = optimize_nQ_intensities(
    nQ_exponential_saturation_model,
    noise_over_S_max = noise_measured / S_max,
    ns=ns, Ns=Ns, M=M,
    I_sat=I_sat, I_max=I_max,
)

## Example 2: Single-N case — per-order error breakdown

When `Ns` has only one element, the function also prints per-order random
and systematic errors, giving insight into which perturbation orders dominate.

In [ ]:
total_err, optimal_I, err_by_order = optimize_nQ_intensities(
    nQ_exponential_saturation_model,
    noise_over_S_max = noise_measured / S_max,
    ns=[1],
    Ns=[4],          # single N → triggers per-order table
    M=4,
    I_sat=I_sat, I_max=I_max,
)
# err_by_order has shape (len(ns), 2*M+1)
print('err_by_order shape:', err_by_order.shape)

## Example 3: Fixed intensities (no optimisation)

If you already know which intensities you will use, pass them as
`fixed_intensities`.  The function evaluates errors at those intensities
without running any optimisation.

In [7]:
# Intensity-cycling intensities with I_0 = 25 nJ, N_total = 6, drop the highest
N_cyc = 6
I_0   = 25       # nJ
I_sat_nJ = 23    # nJ
Is_cyc = 4 * np.cos(np.arange(N_cyc) * np.pi / (2 * N_cyc))**2 * I_0
# Is_cyc = Is_cyc[1:]   # drop the highest intensity

total_err, optimal_I, err_by_order = optimize_nQ_intensities(
    ta_exponential_model,
    noise_over_S_max = 1.6e-6,   # sigma / S_max  (S_max = 1 here)
    ns=[1],
    M=4,
    I_sat=I_sat_nJ,
    fixed_intensities=Is_cyc,
)

Fixed intensities (no optimisation)
  N    1Q tot    1Q rand    1Q sys
---  --------  ---------  --------
  6   0.13768    0.00017   0.13768

Fixed intensities: [100.          93.30127019  75.          50.          25.
   6.69872981]

Errors by perturbation order:
  nQ    r(3)    r(5)    r(7)    r(9)    s(3)    s(5)    s(7)    s(9)
----  ------  ------  ------  ------  ------  ------  ------  ------
   1  0.0000  0.0001  0.0002  0.0003  0.0014  0.0157  0.0821  0.2623


## Example 4: TA signal models

Two built-in models treat the TA signal directly (rather than decomposing
into nQ Bessel components), and are independent of n:

| Model | f(x) | Series term f_series(m, x) |
|---|---|---|
| `ta_exponential_model` | −(1 − e^{−x}) | (−x)^m / m! |
| `ta_saturable_absorption_model` | −x / (x + 1) | (−x)^m |

In [10]:
for model, label in [(ta_exponential_model,         'TA exponential'),
                      (ta_saturable_absorption_model, 'TA saturable absorption')]:
    print(f'\n=== {label} ===')
    optimize_nQ_intensities(
        model,
        noise_over_S_max = 1.6e-6,
        ns=[1],
        Ns=[3, 4, 5, 6],
        M=4,
        I_sat=23.0,   # nJ
        I_max=25.0,   # nJ
    )


=== TA exponential ===
Optimised intensities, I_max = 25.0
  N    1Q tot    1Q rand    1Q sys
---  --------  ---------  --------
  3   0.04689    0.02310   0.04081
  4   0.09695    0.04179   0.08748
  5   0.03434    0.01891   0.02866
  6   0.03681    0.03625   0.00641

Optimal intensities for 1Q (same units as I_sat):
  N     I_1    I_2     I_3     I_4    I_5    I_6
---  ------  -----  ------  ------  -----  -----
  3  0.5959  2.397   3.787
  4  0.9327  4.066   7.516   9.784
  5  1.358   6.142  12.12   17.56   20.74
  6  1.412   6.392  12.8    19.03   23.53     25

=== TA saturable absorption ===
Optimised intensities, I_max = 25.0
  N    1Q tot    1Q rand    1Q sys
---  --------  ---------  --------
  3   0.08158    0.03942   0.07142
  4   0.16907    0.06954   0.15411
  5   0.09046    0.04667   0.07749
  6   0.06119    0.03505   0.05016

Optimal intensities for 1Q (same units as I_sat):
  N     I_1    I_2    I_3    I_4     I_5    I_6
---  ------  -----  -----  -----  ------  -----
  

## Example 5: User-defined saturation model

Supply any model by writing a factory `get_model(n)` that returns
`(f, f_series)` with signatures:

    f(x)          →  signal at dimensionless intensity x = I / I_sat
    f_series(m, x) →  mth-order term in the power-series expansion of f(x)

Below is an example with a Gaussian saturation form
f(x) = −(1 − exp(−x²)), included purely as an illustration of the interface.

In [11]:
from scipy.special import factorial

def gaussian_saturation_model(n):
    """Example user-defined model: f(x) = -(1 - exp(-x^2)).
    Series: f_series(m, x) = (-1)^m * x^(2m) / m!  (for m >= 1, else 0).
    Note: n is ignored here; this model does not have an nQ interpretation.
    """
    def f(x):
        return -(1.0 - np.exp(-x**2))

    def f_series(m, x):
        if m == 0:
            return np.zeros_like(np.asarray(x, dtype=float))
        return (-x**2)**m / factorial(m)

    return f, f_series

total_err, optimal_I, err_by_order = optimize_nQ_intensities(
    gaussian_saturation_model,
    noise_over_S_max = 1e-3,
    ns=[1],
    Ns=[3, 4, 5],
    M=3,
    I_sat=1.0,
    I_max=2.0,
)

Optimised intensities, I_max = 2.0
  N    1Q tot    1Q rand    1Q sys
---  --------  ---------  --------
  3   2.74494    0.83240   2.61569
  4   2.50613    0.97893   2.30703
  5   2.23485    0.85279   2.06574

Optimal intensities for 1Q (same units as I_sat):
  N      I_1     I_2     I_3     I_4    I_5
---  -------  ------  ------  ------  -----
  3  0.0652   0.2686  0.4349
  4  0.07849  0.3367  0.6101  0.7837
  5  0.09101  0.3909  0.7231  0.9527      2


## Example 6: 2Q signal with I0 override

For 2Q signals (n=2), the leading-order series term is zero, so errors
become absolute rather than relative and their scale depends on `I0`.
The default `I0 = I_sat` is usually appropriate, but you can override it.

In [ ]:
# Default: I0 = I_sat
total_err_default, _, _ = optimize_nQ_intensities(
    nQ_exponential_saturation_model,
    noise_over_S_max = 1e-4,
    ns=[2],
    Ns=[3, 4],
    M=3,
    I_sat=14.0,
    I_max=60.0,
    print_tables=False,
)

# Override: I0 = I_sat / 4
total_err_i0, _, _ = optimize_nQ_intensities(
    nQ_exponential_saturation_model,
    noise_over_S_max = 1e-4,
    ns=[2],
    Ns=[3, 4],
    M=3,
    I_sat=14.0,
    I_max=60.0,
    I0=1,     # override I0
    print_tables=False,
)

print('2Q total error with default I0 (= I_sat):')
print(total_err_default[:, [0, 1]])
print('\n2Q total error with I0 = I_sat/4:')
print(total_err_i0[:, [0, 1]])

## Example 7: Suppress tables, work with raw arrays

Set `print_tables=False` to suppress all output and work directly with
the returned arrays.

In [ ]:
total_err, optimal_I, err_by_order = optimize_nQ_intensities(
    nQ_exponential_saturation_model,
    noise_over_S_max = 5e-7 / 0.0543,
    ns=[1],
    Ns=[4],
    M=3,
    I_sat=45.7/60,
    I_max=1.0,
    print_tables=False,
)

# total_err columns: [N, 1Q_tot, 1Q_rand, 1Q_sys]
N_val, tot, rand, sys_ = total_err[0]
print(f'N={int(N_val)}: total={tot:.4f}, random={rand:.4f}, systematic={sys_:.4f}')
print(f'Optimal intensities (in I_sat units): {optimal_I[0, 0, :int(N_val)]}')

In [ ]:
#Luisa's polymer data
# --- Experimental parameters ---
I_sat          = 1             # saturation intensity (dimensionless here; could be nJ)
I_max          = 60/45.7     # maximum achievable intensity
S_max          = 0.0543       # TA signal at saturation
noise_measured = 5e-7         # measured noise amplitude

# --- User choices ---
ns = [1, 2]        # evaluate 1Q and 2Q signals
Ns = [3, 4, 5, 6]  # compare 3-, 4-, 5-, and 6-intensity schemes
M  = 3             # calculate errors using orders 3, 5, 7 (i.e. M perturbation orders)

total_err, optimal_I, err_by_order = optimize_nQ_intensities(
    nQ_exponential_saturation_model,
    noise_over_S_max = noise_measured / S_max,
    ns=ns, Ns=Ns, M=M,
    I_sat=I_sat, I_max=I_max,
)